In [1]:
import anndata as ad
import numpy as np
import os
import sys

SC_PATH = '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated_withoutnans.h5ad'
ST_DIR = '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/results/final/'
OUTPUT_DIR = '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark'

adata = ad.read_h5ad(SC_PATH)


In [2]:
adata.obs['Patient'].unique()

['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']
Categories (7, object): ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']

In [3]:
# Keep the existing patient-level split and use the already harmonized labels.
adata.obs['split'] = 'train'
adata.obs.loc[adata.obs['Patient'].isin(['p5']), 'split'] = 'val'
adata.obs.loc[adata.obs['Patient'].isin(['p1']), 'split'] = 'test'
adata.obs['cell_type'] = adata.obs['cell_type_harmonized']
adata = adata[adata.obs['cell_type_harmonized'] != 'Treg'].copy()
adata.obs['cell_type'] = adata.obs['cell_type_harmonized']
adata.obs['split'].value_counts()


split
train    27144
test      8245
val       4443
Name: count, dtype: int64

In [ ]:
# Each single-cell partition is paired only with its corresponding spatial slices.
partition_slices = {
    'train': ['Lung5_Rep1', 'Lung5_Rep2', 'Lung5_Rep3'],
    'val': ['Lung6'],
    'test': ['Lung9_Rep1', 'Lung9_Rep2', 'Lung12', 'Lung13'],
}

sc_partitions = {
    partition: adata[adata.obs['split'] == partition].copy()
    for partition in partition_slices
}

In [5]:
sys.path.append('/home/jwinchel/Syndri/code')
from mapping import integrate_single_st_file_with_harmony, map_sc_reference_to_unique_xenium_cells

# Integrate each partition with each associated slice, then map the reference
# cells to that slice's unique spatial coordinates.
mapped_by_partition = {partition: [] for partition in partition_slices}

for partition, slices in partition_slices.items():
    sc_partition = sc_partitions[partition]
    if sc_partition.n_obs == 0:
        raise ValueError(f'No single-cell observations in the {partition!r} partition')

    for slice_name in slices:
        st_slice = ad.read_h5ad(os.path.join(ST_DIR, f'st_integrated_slice-{slice_name}.h5ad'))
        st_slice.obs['cell_type'] = st_slice.obs['cell_type_harmonized']
        st_slice = st_slice[st_slice.obs['cell_type'] != 'Treg'].copy()

        sc_integrated, st_integrated, embedding_key = integrate_single_st_file_with_harmony(
            st_slice,
            sc_partition,
            tissue_filter=None,
            n_pcs=30,
            tissue_name=slice_name,
        )
        mapped = map_sc_reference_to_unique_xenium_cells(
            sc_integrated,
            st_integrated,
            embedding_key=embedding_key,
            spatial_cols=('CenterX_global_px', 'CenterY_global_px'),
            n_matches=min(50, sc_integrated.n_obs),
            reuse_penalty=9.0,
            rng=np.random.default_rng(0),
            max_reuse_per_sc=5000,
            match_cell_type=True,
            ensure_all_sc_used=True,
            sc_cell_type_col='cell_type',
            st_cell_type_col='cell_type',
        )
        mapped.obs['split'] = partition
        mapped.obs['partition'] = partition
        mapped.obs['spatial_slice'] = slice_name
        mapped.obs_names = [f'{slice_name}:{cell_id}' for cell_id in mapped.obs_names]
        mapped_by_partition[partition].append(mapped)


/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'cell_type', 'cell_type_harmonized_0', 'cell_type_harmonized_1', 'cell_type_harmonized_2', 'cell_type_harmonized_3', 'cell_type_harmonized_4', 'cell_type_harmonized_5', 'cell_type_harmonized_6', 'cell_type_harmonized_7', 'harmonized_consensus', 'harmonized_agreement', 'cell_type_harmonized', 'split', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_p

2026-10-31 15:00:21,469 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-10-31 15:01:18,297 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-31 15:01:20,979 - harmonypy - INFO - Iteration 1 of 30
2026-10-31 15:05:31,847 - harmonypy - INFO - Iteration 2 of 30
2026-10-31 15:09:30,273 - harmonypy - INFO - Iteration 3 of 30
2026-10-31 15:12:05,537 - harmonypy - INFO - Iteration 4 of 30
2026-10-31 15:13:02,280 - harmonypy - INFO - Iteration 5 of 30
2026-10-31 15:13:56,892 - harmonypy - INFO - Iteration 6 of 30
2026-10-31 15:14:48,448 - harmonypy - INFO - Iteration 7 of 30
2026-10-31 15:15:28,075 - harmonypy - INFO - Iteration 8 of 30
2026-10-31 15:16:20,839 - harmonypy - INFO - Iteration 9 of 30
2026-10-31 15:16:56,481 - harmonypy - INFO - Converged after 9 iterations
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_nam

['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'cell_type', 'cell_type_harmonized_0', 'cell_type_harmonized_1', 'cell_type_harmonized_2', 'cell_type_harmonized_3', 'cell_type_harmonized_4', 'cell_type_harmonized_5', 'cell_type_harmonized_6', 'cell_type_harmonized_7', 'harmonized_consensus', 'harmonized_agreement', 'cell_type_harmonized', 'split', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_p

2026-10-31 15:18:12,455 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-31 15:18:13,201 - harmonypy - INFO - Iteration 1 of 30
2026-10-31 15:19:09,035 - harmonypy - INFO - Iteration 2 of 30
2026-10-31 15:20:04,262 - harmonypy - INFO - Iteration 3 of 30


KeyboardInterrupt: 

In [ ]:
# Combine the mapped spatial slices into the two benchmark files.
train_val = ad.concat(
    mapped_by_partition['train'] + mapped_by_partition['val'],
    join='outer',
    uns_merge='unique',
)
test = ad.concat(
    mapped_by_partition['test'],
    join='outer',
    uns_merge='unique',
)

(train_val.n_obs, test.n_obs)


In [ ]:
# HDF5 serialization requires lists for this mapping metadata field.
for benchmark_data in (train_val, test):
    mapping_metadata = benchmark_data.uns.get('sc_reference_to_spatial_mapping')
    if mapping_metadata and 'spatial_cols' in mapping_metadata:
        mapping_metadata['spatial_cols'] = list(mapping_metadata['spatial_cols'])

os.makedirs(OUTPUT_DIR, exist_ok=True)
train_val.write_h5ad(os.path.join(OUTPUT_DIR, 'train_val.h5ad'))
test.write_h5ad(os.path.join(OUTPUT_DIR, 'test.h5ad'))


## Create the test-set synthetic-admixture inputs

Run this after `test.h5ad` exists. It applies the selected K=9, tau=[0, 0.95] sampler independently within each test slice, excludes Treg cells, and writes the original train/validation single-cell reference once per source cell (without the repeated mapped rows). The resulting files are consumed by the separate RCTD/SPLIT, ResolVI, and Syndri jobs below.

In [3]:
import subprocess

TEST_PIPELINE_DIR = os.path.join(OUTPUT_DIR, 'test_admixture')
TEST_SLICE_DIR = os.path.join(TEST_PIPELINE_DIR, 'slices')
os.makedirs(TEST_SLICE_DIR, exist_ok=True)
subprocess.run([
    sys.executable,
    '/home/jwinchel/Syndri/scripts/spillover-benchmark/prepare_test_admixture.py',
    '--test', os.path.join(OUTPUT_DIR, 'test.h5ad'),
    '--sc-reference', SC_PATH,
    '--output-dir', TEST_SLICE_DIR,
    '--reference-output', os.path.join(TEST_PIPELINE_DIR, 'train_val_sc_reference_unique.h5ad'),
], check=True)


beginning admixture sampling...
admixture sampled!
/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/slices/Lung12.h5ad: cells=71,286, genes=956, mean achieved burden=0.5241
beginning admixture sampling...
admixture sampled!
/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/slices/Lung13.h5ad: cells=78,992, genes=956, mean achieved burden=0.5162
beginning admixture sampling...
admixture sampled!
/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/slices/Lung9_Rep1.h5ad: cells=86,337, genes=956, mean achieved burden=0.5237
beginning admixture sampling...
admixture sampled!
/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/slices/Lung9_Rep2.h5ad: cells=137,788, genes=956, mean achieved burden=0.5298
Unique train/val SC reference: /work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/train_val

CompletedProcess(args=['/home/jwinchel/miniconda3/envs/rctd/bin/python', '/home/jwinchel/Syndri/scripts/spillover-benchmark/prepare_test_admixture.py', '--test', '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test.h5ad', '--sc-reference', '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated_withoutnans.h5ad', '--output-dir', '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/slices', '--reference-output', '/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/spillover-benchmark/test_admixture/train_val_sc_reference_unique.h5ad'], returncode=0)

## Run correction methods on the test slices

Once the preparation cell completes, submit the per-slice method jobs from a login node:

```bash
bash /home/jwinchel/Syndri/scripts/spillover-benchmark/submit_test_benchmark.sh
```

The script uses the unique train/validation SC reference for RCTD and SPLIT, fits ResolVI separately for each slice, and runs the trained Syndri checkpoint at milestone 29. Syndri writes per-gene admixture probabilities and a hard-threshold correction layer (`syndri_corrected_counts`, probability threshold 0.5); that threshold is configurable in `run_syndri_test.sbatch`. Outputs are saved below `test_admixture/`.